# 01 - Data Collection dan Understanding (FR-01)
**Input:** `data/raw/bank-full.csv` · **Output:** ringkasan validasi

- [x] Cek file ada (unduh dari UCI jika belum)
- [x] Baca CSV (delimiter `;`) dan validasi kolom wajib
- [x] Tampilkan jumlah baris/kolom, tipe data, missing, duplikat, distribusi `y`
- [x] Tulis catatan pemahaman dataset (hanya dari output aktual)


In [1]:
# --- Setup path (jalan di VS Code / Jupyter. Untuk Colab, lihat sel opsional di bawah) ---
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_PATH = ROOT / "data" / "raw" / "bank-full.csv"
CLEAN_PATH = ROOT / "data" / "processed" / "bank_campaign_clean.csv"
MODELS_DIR = ROOT / "models"
SQL_DIR = ROOT / "sql"
DASHBOARD_PATH = ROOT / "reports" / "bank_campaign_dashboard.xlsx"
print("ROOT:", ROOT)

ROOT: .


In [2]:
# (Opsional) Hanya jika dijalankan di Google Colab:
# from google.colab import drive
# drive.mount('/content/drive')
# import os; os.chdir('/content/drive/MyDrive/bank-campaign-response-analytics')  # sesuaikan lokasi folder
# Jalankan sel Setup di atas SETELAH baris ini.

In [3]:
REQUIRED_COLUMNS = ["age", "job", "marital", "education", "default", "balance", "housing", "loan",
                    "contact", "day", "month", "duration", "campaign", "pdays", "previous", "poutcome", "y"]
TARGET = "y"
RANDOM_STATE = 42

## 1. Baca data
Dataset UCI memakai pemisah titik koma (`;`), bukan koma.

In [4]:
if not RAW_PATH.exists():
    raise FileNotFoundError(f"{RAW_PATH.name} tidak ditemukan. Unduh dari UCI dan simpan di data/raw/.")

df = pd.read_csv(RAW_PATH, sep=";")
if df.shape[1] == 1:
    raise ValueError("Jumlah kolom tidak sesuai. Periksa delimiter (diharapkan ';').")
df.columns = [c.strip().lower() for c in df.columns]
print("Jumlah baris, kolom:", df.shape)
df.head()

Jumlah baris, kolom: (45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


## 2. Validasi

In [5]:
missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
assert not missing_cols, f"Kolom wajib hilang: {missing_cols}"

classes = set(df[TARGET].dropna().unique())
assert len(classes) == 2, "Target harus punya dua kelas"
assert classes <= {"yes", "no"}, f"Nilai target tak terduga: {classes}"
print("Kolom lengkap dan target valid")

Kolom lengkap dan target valid


In [6]:
df.dtypes

age          int64
job            str
marital        str
education      str
default        str
balance      int64
housing        str
loan           str
contact        str
day          int64
month          str
duration     int64
campaign     int64
pdays        int64
previous     int64
poutcome       str
y              str
dtype: object

In [7]:
print("Missing values per kolom:")
print(df.isna().sum())
print("\nBaris duplikat:", df.duplicated().sum())

Missing values per kolom:
age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64

Baris duplikat: 0


In [8]:
print(df[TARGET].value_counts())
print(df[TARGET].value_counts(normalize=True).round(4))

y
no     39922
yes     5289
Name: count, dtype: int64
y
no     0.883
yes    0.117
Name: proportion, dtype: float64


## Catatan Pemahaman Dataset

Berdasarkan output sel di atas:

| Atribut | Nilai |
|---|---|
| Jumlah baris | 45.211 |
| Jumlah kolom | 17 |
| Kelas target `yes` | 5.289 (11,70%) |
| Kelas target `no` | 39.922 (88,30%) |
| Missing values | 0 (tidak ada NaN eksplisit) |
| Baris duplikat | 0 |

### Struktur Kolom
- **Numerik (7):** `age`, `balance`, `day`, `duration`, `campaign`, `pdays`, `previous`
- **Kategorikal (9):** `job`, `marital`, `education`, `default`, `housing`, `loan`, `contact`, `month`, `poutcome`
- **Target (1):** `y` (`yes`/`no`)

### Observasi Kunci
- Dataset menggunakan delimiter titik koma (`;`), bukan koma
- Kolom `pdays = -1` menandakan nasabah **belum pernah** dihubungi pada kampanye sebelumnya
  (36.954 dari 45.211 baris = 81,7%)
- Nilai `unknown` hadir di beberapa kolom kategorikal:
  `poutcome` (36.959), `contact` (13.020), `education` (1.857), `job` (288)
- **Ketidakseimbangan kelas** signifikan: 88,3% 'no' vs 11,7% 'yes' —
  evaluasi model harus menggunakan Recall, F1, dan PR-AUC, bukan Accuracy
- Kolom `duration` (durasi panggilan terakhir) hanya diketahui **setelah** panggilan berlangsung;
  tidak boleh digunakan sebagai fitur model prediksi sebelum panggilan (data leakage)

### Keterbatasan Dataset
- Data kampanye bank di **Portugal**, bukan Indonesia
- Merupakan data kampanye telemarketing historis; kondisi pasar saat ini mungkin berbeda
- Proyek ini adalah demonstrasi analitik portofolio, **bukan rekomendasi untuk keputusan perbankan produksi**
